### Import

In [1]:
import gc
import os
import sys
import re
import math
import numpy as np
import pandas as pd
import datetime as dt
from tqdm import tqdm
from matplotlib import pyplot as plt 

import warnings
from collections import Counter

from multiprocessing import Pool
from multiprocessing import cpu_count

import pywt
import librosa 
from scipy.fftpack import fft
from scipy.signal import stft

from scipy import stats
from scipy import integrate

from scipy.stats import skew 
from scipy.stats import kurtosis
from scipy.stats import median_abs_deviation

from scipy.signal import welch
from scipy.signal import hamming

pd.set_option('display.max_columns', 500)

### General parameters

In [2]:
path_timeseries = '../Data/Output/'

### Read Data

In [3]:
def dataframe_from_csv(path, fields, header=0, index_col=False):
    return pd.read_csv(path, usecols=fields, header=header, index_col=index_col)

### Range Filter

In [4]:
def set_range(signal, Range_min=50, Range_max=100):

    signal = np.array(signal)
    signal = np.ma.masked_where(((signal < Range_min) | (signal > Range_max)), signal).tolist()
    signal = [x if x is not None else np.nan for x in signal]
    signal = np.array(signal)
    
    return signal

### Delta Filter

In [5]:
def delta_filter(signal, Diff=15):

    signal_filtered = np.zeros_like(signal)
    signal_filtered[0] = signal[0]
    
    for i in range(1, len(signal)):

        if np.isnan(signal[i]):
            signal_filtered[i] = np.nan
            continue
        elif np.isnan(signal_filtered[i-1]):
            signal_filtered[i] = signal[i]
            continue
        
        rel_diff = abs((signal_filtered[i-1] - signal[i]) / signal_filtered[i-1]) * 100
        
        if rel_diff < Diff:
            signal_filtered[i] = signal[i]
        else:
            signal_filtered[i] = np.nan
            
    return signal_filtered

### Block Data Filter

In [6]:
def block_data_filter(signal, threshold=50, block_size=100, block_mean_ratio=0.94):
    
    signal = np.array(signal)
    mask = np.ones(len(signal), dtype=bool)
    
    for i, data in enumerate(signal):
        if data < threshold:
            mask[max(0, i - 10):min(len(signal), i + 10)] = False

    mean_signal = np.nanmean(signal)
    i = 0
    while i + block_size < len(signal):
        mean_block = np.nanmean(signal[i:i + block_size])
        if mean_block < mean_signal * block_mean_ratio:
            mask[i:i + block_size] = False
        i += block_size
    
    signal[~mask] = np.nan

    return signal

### Resample Signal

In [7]:
def resamp_spo2(signal, OriginalFreq):

    len_in = len(signal)
    len_out = int(round(len_in / OriginalFreq))
    data_out = np.zeros(len_out)
    
    for jj in range(len_out):
        start_idx = int(jj * OriginalFreq)
        end_idx = int((jj + 1) * OriginalFreq)
        
        slice_data = signal[start_idx:end_idx]
        
        if np.all(np.isnan(slice_data)):
            data_out[jj] = np.nan
        else:
            data_out[jj] = np.nanmedian(slice_data)
        
    return data_out

### Overall General Measures 

In [8]:
def delta_index(signal, DI_Window):

    if DI_Window >= len(signal):
        raise ValueError("DI_Window should be less than the length of the signal.")

    signal_splitted = [signal[i:i + DI_Window] for i in range(0, len(signal), DI_Window)]
    if len(signal_splitted[-1]) != DI_Window:
        signal_splitted.pop()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        mean_window = np.nanmean(signal_splitted, axis=1)
    diff = abs(mean_window - np.roll(mean_window, 1))
    
    return np.round(np.nanmean(diff[1:]), 2)

In [9]:
def num_zc(signal, ZC_Baseline):

    numZC_count = 0
    baseline = ZC_Baseline
    for idx_signal in range(2, len(signal) - 1):
        if signal[idx_signal] == baseline:
            if (signal[idx_signal - 1] <= baseline) & (signal[idx_signal + 1] >= baseline):
                numZC_count += 1
            if (signal[idx_signal - 1] >= baseline) & (signal[idx_signal + 1] <= baseline):
                numZC_count += 1
        if (signal[idx_signal - 1] < baseline) & (signal[idx_signal] > baseline):
            numZC_count += 1
        if (signal[idx_signal - 1] > baseline) & (signal[idx_signal] < baseline):
            numZC_count += 1
            
    return numZC_count

In [10]:
def below_median(signal, M_Threshold):

    baseline = np.nanmedian(signal) - M_Threshold
    
    with np.errstate(invalid='ignore'):
        return np.round(100 * (np.nansum(signal < baseline) / len(signal)), 2)

In [11]:
def above_median(signal, M_Threshold):

    baseline = np.nanmedian(signal) + M_Threshold
    
    with np.errstate(invalid='ignore'):
        return np.round(100 * (np.nansum(signal > baseline) / len(signal)), 2)

In [12]:
def apply_percentile(signal, percentile):
 
    return np.nanpercentile(signal, percentile)

In [13]:
def compute_range(signal):

    return np.nanmax(signal) - np.nanmin(signal)

In [14]:
def check_shape(signal):
    
    assert len(signal) > 0 
    signal = np.array(signal)
    assert len(signal.shape) == 1

In [15]:
def OverallGeneralMeasures(signal, ZC_Baseline=None, M_Threshold=2, DI_Window=12):

    check_shape(signal)

    if ZC_Baseline is None:
        ZC_Baseline = np.nanmean(signal)
        
    warnings.simplefilter("ignore")

    measures = {
                'AV' : np.round(np.nanmean(signal), 2),
                'MED': np.nanmedian(signal),
                'Min': np.nanmin(signal),
                'Max': np.nanmax(signal),
                'SD' : np.round(np.nanstd(signal), 2),
                'KS' : np.round(kurtosis(signal[~np.isnan(signal)]), 2),
                'SK' : np.round(float(skew(signal[~np.isnan(signal)], axis=None)), 2),
                'MAD': np.round(median_abs_deviation(signal[~np.isnan(signal)]), 2),
                'RG' : compute_range(signal),
                'P25': apply_percentile(signal, 25),
                'P75': apply_percentile(signal, 75),
                'BM' : below_median(signal, M_Threshold),
                'AM' : above_median(signal, M_Threshold),
                'ZC' : num_zc(signal, ZC_Baseline),
                'DI' : delta_index(signal, DI_Window)
               }

    return measures

### Periodicity - PRSA

In [16]:
def PRSAMeasures(signal, PRSA_Window=10, K_AC=2):

    check_shape(signal)

    assert PRSA_Window > 0

    d = PRSA_Window
    anchor_points = []
    anchor_found = False

    for i in range(len(signal)):
        if i < d:
            continue
        if len(signal) - i < d:
            continue
        if signal[i - 1] > signal[i]:
            anchor_found = True
            anchor_points.append(signal[i - d:i + d])

    if anchor_found is False:
        PRSAc  = 0
        PRSAad = 0
        PRSAos = 0
        PRSAsb = 0
        PRSAsa = 0
        AC = np.round(np.correlate(signal, signal, "same")[K_AC], 2)

    else:
        anchor_points = np.array(anchor_points)
        windows = np.zeros(2 * d)

        for i in range(2 * d):
            windows[i] = np.nansum(anchor_points[:, i]) / len(anchor_points)

        PRSAc  = np.round((windows[d] + windows[d + 1] - windows[d - 1] - windows[d - 2]) / 4, 2)
        PRSAad = np.round(np.nanmax(windows) - np.nanmin(windows), 2)
        PRSAos = np.round(float(np.polyfit(range(2 * d), windows, 1)[0]), 2)
        PRSAsb = np.round(float(np.polyfit(range(d), windows[0:d], 1)[0]), 2)
        PRSAsa = np.round(float(np.polyfit(range(d), windows[d:], 1)[0]), 2)
        AC = np.round(np.correlate(windows, windows, "same")[K_AC], 2)

    measures = {
                'PRSAc' : PRSAc,
                'PRSAad': PRSAad,
                'PRSAos': PRSAos,
                'PRSAsb': PRSAsb,
                'PRSAsa': PRSAsa,
                'AC': AC,  
               }

    return measures

### Periodicity - PSD

In [17]:
def get_bandpass(signal, freq, lower_f, higher_f):

    amplitude_bp = signal[lower_f < freq]
    freq_bp = freq[lower_f < freq]

    amplitude_bp = amplitude_bp[freq_bp < higher_f]
    
    return amplitude_bp

In [18]:
def get_psd(signal):

    freq, signal_fft = welch(signal, fs=0.2, window="hamming")
    signal_fft = signal_fft / len(signal)

    return freq, signal_fft

In [19]:
def PSDMeasures(signal, frequency_low_threshold= 0.014, frequency_high_threshold= 0.033):
 
    check_shape(signal)
    
    assert frequency_low_threshold <= frequency_high_threshold
    
    signal = np.array(signal)
    signal = signal[np.logical_not(np.isnan(signal))]

    freq, signal_fft = get_psd(signal)
    amplitude_signal = np.sqrt((signal_fft.real ** 2) + (signal_fft.imag ** 2))

    freq = freq[0:int(len(freq) / 2)]
    amplitude_signal = amplitude_signal[0:int(len(amplitude_signal) / 2)]

    amplitude_bp = get_bandpass(amplitude_signal, freq, frequency_low_threshold, frequency_high_threshold)

    if len(amplitude_bp) == 0:
        
        PSD_total = np.round(np.nansum(amplitude_signal), 2)
        PSD_band  = 0
        PSD_ratio = 0
        PSD_peak  = 0
        
    else:
        PSD_total = np.round(np.nansum(amplitude_signal), 2)
        PSD_band  = np.round(np.nansum(amplitude_bp), 2)
        PSD_ratio = np.round(np.nansum(amplitude_bp) / np.nansum(amplitude_signal), 2)
        PSD_peak  = np.round(float(np.nanmax(amplitude_bp)), 2)

    measures = {
                'PSD_total': PSD_total,
                'PSD_band' : PSD_band,
                'PSD_ratio': PSD_ratio,
                'PSD_peak' : PSD_peak,
               }

    return measures

### Complexity Measures

In [20]:
def comp_dfa(signal, DFA_Window):

    n = DFA_Window
    y = integrate.cumtrapz(signal - np.nanmean(signal))
    least_square = np.zeros(len(y))

    i = 0
    while i < len(y):
        if i + n > len(y):
            n = len(y) - i
        if n == 1:
            least_square[-1] = y[-1]
            break
        x = np.array(range(0, n))
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            slope, intercept, _, _, _ = stats.linregress(x, y[i:i + n])
        least_square[i:i + n] = slope * x + intercept
        i += n

    return np.sqrt(np.nansum((y - least_square) ** 2) / len(y))

In [21]:
def comp_sampen(signal, M_Sampen, R_Sampen):

    N = len(signal)
    m = M_Sampen
    r = R_Sampen

    xmi = np.array([signal[i: i + m] for i in range(N - m)])
    xmj = np.array([signal[i: i + m] for i in range(N - m + 1)])

    with np.errstate(invalid='ignore'):
        B = np.sum([np.sum(np.abs(xmii - xmj).max(axis=1) <= r) - 1 for xmii in xmi])

    m += 1
    xm = np.array([signal[i: i + m] for i in range(N - m + 1)])

    with np.errstate(invalid='ignore'):
        A = np.sum([np.sum(np.abs(xmi - xm).max(axis=1) <= r) - 1 for xmi in xm])

    if A == 0:
        return 0
    elif B == 0:
        return 0
    else:
        return -np.log(A / B)

In [22]:
def d_ctm(i, p, signal):

    if np.sqrt(((signal[i + 2] - signal[i + 1]) ** 2) + ((signal[i + 1] - signal[i]) ** 2)) < p:
        return 1
    return 0

In [23]:
def comp_ctm(signal, CTM_Threshold):

    res = 0
    for i in range(len(signal) - 2):
        res += d_ctm(i, CTM_Threshold, signal)
        
    return res / (len(signal) - 2)

In [24]:
def comp_lz(signal, dual_quantization=True):

    signal = np.array(signal)

    if dual_quantization is True:
        median = np.nanmedian(signal)
        bool_median = signal > median
        byte = [str(int(b == bool(True))) for b in bool_median]
    else:
        quantile1, median, quantile3 = np.quantile(signal, 0.25), np.quantile(signal, 0.5), np.quantile(signal, 0.75)
        byte = np.empty(shape=signal.shape, dtype=str)
        byte[signal <= quantile1] = "0"
        byte[(signal > quantile1) & (signal <= median)] = "1"
        byte[(signal > median) & (signal <= quantile3)] = "2"
        byte[signal > quantile3] = "3"

    sequence = ''.join(byte)

    sub_strings = set()

    ind = 0
    inc = 1
    while True:
        if ind + inc > len(sequence):
            break
        sub_str = sequence[ind: ind + inc]
        if sub_str in sub_strings:
            inc += 1
        else:
            sub_strings.add(sub_str)
            ind += inc
            inc = 1
            
    return len(sub_strings)

In [25]:
def dist(window1, window2, r):

    window1 = np.array(window1)
    window2 = np.array(window2)

    with np.errstate(invalid='ignore'):
        return np.nanmax(abs(window1 - window2), axis=1) < r

In [26]:
def apen(M_ApEn, R_ApEn, signal):
     
    m = M_ApEn
    r = R_ApEn
    N = len(signal)
    
    if N - m + 1 < 0:
        return np.nan
            
    C = np.zeros(shape=(N - m + 1))
    res = [signal[i:i + m] for i in range(0, N - m + 1)]
    
    for i in range(0, N - m + 1):
        C[i] = np.nansum(dist(res[i], res, r)) / (N - m + 1)

    phi_m = np.nansum(np.log(C)) / (N - m + 1)
    
    return phi_m

In [27]:
def comp_apen(signal, M_ApEn, R_ApEn):

    signal = np.array(signal)
    signal = signal[np.logical_not(np.isnan(signal))]

    phi_m  = apen(M_ApEn, R_ApEn, signal)
    phi_m1 = apen(M_ApEn + 1, R_ApEn, signal)
    
    with np.errstate(invalid='ignore'):
        res = phi_m - phi_m1
        
    return res

In [28]:
def ComplexityMeasures(signal, CTM_Threshold=0.25, DFA_Window=12, M_Sampen=3, R_Sampen=0.2, M_ApEn=2, R_ApEn=0.25):

    check_shape(signal)
    
    assert DFA_Window > 0
    assert M_Sampen > 0
    assert R_Sampen > 0
    assert M_ApEn   > 0
    assert R_ApEn   > 0
    
    ApEn = comp_apen(signal, M_ApEn, R_ApEn)
    LZ   = comp_lz(signal)
    CTM  = comp_ctm(signal, CTM_Threshold)
    SampEn = comp_sampen(signal, M_Sampen, R_Sampen)
    DFA  = comp_dfa(signal, DFA_Window)
    
    ApEn = np.round(ApEn, 2)
    LZ   = np.round(LZ, 2)
    CTM  = np.round(CTM, 2)
    SampEn = np.round(SampEn, 2)
    DFA  = np.round(DFA, 2)
    
    measures = {
                'ApEn': ApEn,
                'LZ'  : LZ,
                'CTM' : CTM,
                'SampEn': SampEn,
                'DFA' : DFA
               }
    
    return measures

### Desaturations Measures

In [29]:
def DesaturationsMeasuresResults(ODI, DL_u, DL_sd, DA100_u, DA100_sd, DAmax_u, DAmax_sd,
                                 DD100_u, DD100_sd, DDmax_u, DDmax_sd, DS_u, DS_sd,
                                 TD_u, TD_sd, DL_a_u, DL_a_sd, DL_b_u, DL_b_sd,
                                 begin_desat, end_desat,
                                 lowerbound, upperbound):
    
    begin_desat = np.array(begin_desat) if isinstance(begin_desat, list) else begin_desat
    end_desat   = np.array(end_desat) if isinstance(end_desat, list) else end_desat
    
    if lowerbound:
        measures = {
                'LP_ODI': ODI,
                'LP_DL_u'  : DL_u,
                'LP_DL_sd' : DL_sd,
                'LP_DA100_u': DA100_u,
                'LP_DA100_sd' : DA100_sd,
                'LP_DAmax_u': DAmax_u,
                'LP_DAmax_sd': DAmax_sd,
                'LP_DD100_u': DD100_u,
                'LP_DD100_sd': DD100_sd,
                'LP_DDmax_u': DDmax_u,
                'LP_DDmax_sd': DDmax_sd,
                'LP_DS_u': DS_u,
                'LP_DS_sd': DS_sd,
                'LP_TD_u': TD_u,
                'LP_TD_sd': TD_sd,
                'LP_DL_a_u': DL_a_u,
                'LP_DL_a_sd': DL_a_sd,
                'LP_DL_b_u': DL_b_u,
                'LP_DL_b_sd': DL_b_sd,
                'LP_begin': begin_desat.flatten().tolist(),
                'LP_end': end_desat.flatten().tolist()
               }
        
    elif upperbound:
        measures = {
                'UP_ODI': ODI,
                'UP_DL_u'  : DL_u,
                'UP_DL_sd' : DL_sd,
                'UP_DA100_u': DA100_u,
                'UP_DA100_sd' : DA100_sd,
                'UP_DAmax_u': DAmax_u,
                'UP_DAmax_sd': DAmax_sd,
                'UP_DD100_u': DD100_u,
                'UP_DD100_sd': DD100_sd,
                'UP_DDmax_u': DDmax_u,
                'UP_DDmax_sd': DDmax_sd,
                'UP_DS_u': DS_u,
                'UP_DS_sd': DS_sd,
                'UP_TD_u': TD_u,
                'UP_TD_sd': TD_sd,
                'UP_DL_a_u': DL_a_u,
                'UP_DL_a_sd': DL_a_sd,
                'UP_DL_b_u': DL_b_u,
                'UP_DL_b_sd': DL_b_sd,
                'UP_begin': begin_desat.flatten().tolist(),
                'UP_end': end_desat.flatten().tolist()
               }            
    
    return measures

In [30]:
def desat_embedding(begin_desat, end_desat, minmax_desat):

    table_desat_aa = begin_desat
    table_desat_cc = end_desat

    if isinstance(table_desat_aa, int):
        table_desat_aa = [table_desat_aa]
    if isinstance(minmax_desat, int):
        minmax_desat = [minmax_desat]
    if isinstance(table_desat_cc, int):
        table_desat_cc = [table_desat_cc]

    desaturations = []  
    for kk in range(0, len(table_desat_aa)):
        desaturations.append({
            'Start': int(table_desat_aa[kk]),
            'Duration': table_desat_cc[kk] - table_desat_aa[kk],
            'End': int(table_desat_cc[kk]),
            'Min_to_Begin': minmax_desat[kk] - table_desat_aa[kk],
            'End_to_Min': table_desat_cc[kk] - minmax_desat[kk],
        })

    desaturation_valid = np.full(len(desaturations), False)
    desaturation_length_all  = np.full(len(desaturations), np.nan)
    desaturation_int_100_all = np.full(len(desaturations), np.nan)
    desaturation_int_max_all = np.full(len(desaturations), np.nan)
    desaturation_depth_100_all = np.full(len(desaturations), np.nan)
    desaturation_depth_max_all = np.full(len(desaturations), np.nan)
    desaturation_slope_all  = np.full(len(desaturations), np.nan)
    desaturations_min_begin = np.full(len(desaturations), np.nan)
    desaturations_end_min   = np.full(len(desaturations), np.nan)
    
    return desaturations, desaturation_valid, desaturation_length_all, desaturation_int_100_all, \
           desaturation_int_max_all, desaturation_depth_100_all, desaturation_depth_max_all, desaturation_slope_all, \
           desaturations_min_begin, desaturations_end_min

In [31]:
def get_desaturation_features(signal, begin_desat, end_desat, minmax_desat, lowerbound, upperbound):
    
    if len(begin_desat) == 0 or len(end_desat) == 0:
        return DesaturationsMeasuresResults(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, [], [],
                                            lowerbound, upperbound)

    ODI = len(begin_desat) / len(signal) * 12

    desaturations, desaturation_valid, desaturation_length_all, desaturation_int_100_all, \
    desaturation_int_max_all, desaturation_depth_100_all, desaturation_depth_max_all, \
    desaturation_slope_all, desaturations_min_begin, desaturations_end_min = \
        desat_embedding(begin_desat, end_desat, minmax_desat)

    time_spo2_array = np.array(range(len(signal)))

    starts = []
    for (i, desaturation) in enumerate(desaturations):
        desaturation_idx = (time_spo2_array >= desaturation['Start']) & (time_spo2_array <= desaturation['End'])

        if np.sum(desaturation_idx) == 0:
            continue

        starts.append(desaturation['Start'])
        signal = np.array(signal)

        desaturation_time = time_spo2_array[desaturation_idx]
        desaturation_spo2 = signal[desaturation_idx]
        desaturation_min  = np.nanmin(desaturation_spo2)
        desaturation_max  = np.nanmax(desaturation_spo2)

        desaturation_valid[i] = True

        desaturation_length_all[i] = desaturation['Duration']
        desaturations_min_begin[i] = desaturation['Min_to_Begin']
        desaturations_end_min[i]   = desaturation['End_to_Min']

        desaturation_int_100_all[i]   = np.nansum(100 - desaturation_spo2)
        desaturation_int_max_all[i]   = np.nansum(desaturation_max - desaturation_spo2)
        desaturation_depth_100_all[i] = 100 - desaturation_min
        desaturation_depth_max_all[i] = desaturation_max - desaturation_min

        desaturation_idx_max = np.where(desaturation_spo2 == desaturation_max)[0][0]  
        desaturation_idx_min = np.where(desaturation_spo2 == desaturation_min)[0][-1] 
        desaturation_idx_max_min = np.arange(desaturation_idx_max, desaturation_idx_min + 1)

        if len(desaturation_idx_max_min) > 0:
            try:
                p = np.polyfit(np.int64(desaturation_time[desaturation_idx_max_min]),
                               desaturation_spo2[desaturation_idx_max_min], 1)
                desaturation_slope_all[i] = p[0]
            except:
                desaturation_slope_all[i] = np.nan

    diff_desats = abs(starts - np.roll(starts, 1))
    diff_desats = diff_desats[1:]

    begin_desat = np.array(begin_desat)
    end_desat   = np.array(end_desat)

    if np.sum(desaturation_valid) != 0:
        
        DL_u     = np.round(float(np.nanmean(desaturation_length_all[desaturation_valid])), 2)
        DL_sd    = np.round(float(np.nanstd(desaturation_length_all[desaturation_valid])) , 2)
        DA100_u  = np.round(float(np.nanmean(desaturation_int_100_all[desaturation_valid])), 2)
        DA100_sd = np.round(float(np.nanstd(desaturation_int_100_all[desaturation_valid])) , 2)
        DAmax_u  = np.round(float(np.nanmean(desaturation_int_max_all[desaturation_valid])), 2)
        DAmax_sd = np.round(float(np.nanstd(desaturation_int_max_all[desaturation_valid])) , 2)
        DD100_u  = np.round(float(np.nanmean(desaturation_depth_100_all[desaturation_valid])), 2)
        DD100_sd = np.round(float(np.nanstd(desaturation_depth_100_all[desaturation_valid])) , 2)
        DDmax_u  = np.round(float(np.nanmean(desaturation_depth_max_all[desaturation_valid])), 2)
        DDmax_sd = np.round(float(np.nanstd(desaturation_depth_max_all[desaturation_valid])) , 2)

        DS_u  = np.round(float(np.nanmean(desaturation_slope_all[desaturation_valid])), 2)
        DS_sd = np.round(float(np.nanstd(desaturation_slope_all[desaturation_valid])) , 2)
        TD_u  = np.round(float(np.nanmean(diff_desats)), 2)
        TD_sd = np.round(float(np.nanstd(diff_desats)), 2)
        
        DL_a_u  = np.round(float(np.nanmean(desaturations_min_begin[desaturation_valid])), 2)
        DL_a_sd = np.round(float(np.nanstd(desaturations_min_begin[desaturation_valid])) , 2)
        DL_b_u  = np.round(float(np.nanmean(desaturations_end_min[desaturation_valid])), 2)
        DL_b_sd = np.round(float(np.nanstd(desaturations_end_min[desaturation_valid])) , 2)
        
        ODI = np.round(ODI, 2)

        desaturation_features = DesaturationsMeasuresResults(ODI, DL_u, DL_sd, DA100_u, DA100_sd, DAmax_u, DAmax_sd,
                                                             DD100_u, DD100_sd, DDmax_u, DDmax_sd, DS_u, DS_sd,
                                                             TD_u, TD_sd, DL_a_u, DL_a_sd, DL_b_u, DL_b_sd,
                                                             begin_desat, end_desat, lowerbound, upperbound)
    else:
        desaturation_features = DesaturationsMeasuresResults(ODI, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
                                                             0, begin_desat, end_desat, lowerbound, upperbound)

    return desaturation_features

In [32]:
def remove_small_desats(counter_desat, begin_desat, end_desat, minmax_desat, desat_min_length):
    
    new_begin = [] 
    new_end = [] 
    new_minmax = [] 
    new_counter = []
    
    for i_begin, i_end, i_min, i_counter in zip(begin_desat, end_desat, minmax_desat, counter_desat):
        if i_end - i_begin > desat_min_length:
            new_begin.append(i_begin)
            new_end.append(i_end)
            new_minmax.append(i_min)
            new_counter.append(i_counter)

    begin_desat = new_begin
    end_desat = new_end
    minmax_desat = new_minmax
    counter_desat = new_counter
    
    return counter_desat, begin_desat, end_desat, minmax_desat

In [33]:
def remove_nan_desats(signal, begin_desat, end_desat, minmax_desat, counter_desat):

    idx = np.array([True]*len(begin_desat))
    for i, (i_begin, i_end) in enumerate(zip(begin_desat, end_desat)):
        idx[i] = ~np.all(np.isnan(signal[i_begin:i_end]))
        
    if not (len(begin_desat) and len(end_desat) and len(minmax_desat) and len(counter_desat)):
        return counter_desat, begin_desat, end_desat, minmax_desat

    begin_desat = begin_desat[idx]
    end_desat = end_desat[idx]
    minmax_desat = [minmax_desat[i] for i,x in enumerate(idx) if x]
    counter_desat = [counter_desat[i] for i,x in enumerate(idx) if x]
    
    return counter_desat, begin_desat, end_desat, minmax_desat

In [34]:
def group_meta_desat(signal, min_dist_meta_event, begin_desat, end_desat, minmax_desat,
                     lowerbound= True, upperbound= False):
    
    end_before = - min_dist_meta_event - 5

    new_idx_desat = np.zeros(shape=(len(begin_desat)), dtype=np.int8)
    count_desat, count_meta_desat = 0, -1

    for begin_index, end_index in zip(begin_desat, end_desat):
        if begin_index - end_before > min_dist_meta_event:
            count_meta_desat += 1

        new_idx_desat[count_desat] = count_meta_desat
        end_before = end_index
        count_desat += 1

    curr_meta_desat = 0
    begin_inserted = False
    new_begin = np.zeros(shape=(len(begin_desat))) 
    new_end   = np.zeros(shape=(len(begin_desat)))
    new_minmax= np.zeros(shape=(len(begin_desat)))
    
    if lowerbound:     
        for idx_desat, idx_meta_desat in enumerate(new_idx_desat):
            if idx_meta_desat != curr_meta_desat:
                curr_meta_desat += 1
                begin_inserted = False

            if begin_inserted is False:
                new_begin[idx_meta_desat] = begin_desat[idx_desat]
                begin_inserted = True
            new_end[idx_meta_desat] = max(end_desat[idx_desat], int(new_end[idx_meta_desat]))
            new_minmax[idx_meta_desat] = new_begin[idx_meta_desat] + \
                                         np.argmin(signal[int(new_begin[idx_meta_desat]): int(new_end[idx_meta_desat])])

    elif upperbound:
        for idx_desat, idx_meta_desat in enumerate(new_idx_desat):
            if idx_meta_desat != curr_meta_desat:
                curr_meta_desat += 1
                begin_inserted = False

            if begin_inserted is False:
                new_begin[idx_meta_desat] = begin_desat[idx_desat]
                begin_inserted = True
            new_end[idx_meta_desat] = max(end_desat[idx_desat], int(new_end[idx_meta_desat]))
            new_minmax[idx_meta_desat] = new_begin[idx_meta_desat] + \
                                         np.argmax(signal[int(new_begin[idx_meta_desat]): int(new_end[idx_meta_desat])])       
        
    begin_desat = new_begin[new_begin != 0].astype(int)
    end_desat   = new_end[new_end != 0].astype(int)
    minmax_desat= new_minmax[new_minmax != 0].astype(int)

    counter_desat = Counter(new_idx_desat)
    counter_desat = list(counter_desat.values())
    
    return counter_desat, begin_desat, end_desat, minmax_desat

In [35]:
def hard_threshold_detector(signal, threshold_method, hard_threshold, quantile_threshold,
                            lowerbound= True, upperbound= False):

    if threshold_method == 'Hard':
        threshold_value = hard_threshold
    elif threshold_method == 'Quantile':
        threshold_value = np.nanquantile(signal, quantile_threshold)

    begin_desat= [] 
    end_desat  = []
    minmax_desat  = []
    
    
    turn_begin = True
    
    if lowerbound:
        for i in range(len(signal)):
            if i == 0:
                continue
            if np.isnan(signal[i]):  
                continue
            if (signal[i - 1] >= threshold_value) and (signal[i] < threshold_value):
                if turn_begin is True:
                    begin_desat.append(i)
                    turn_begin = False
            if (signal[i - 1] < threshold_value) and (signal[i] >= threshold_value):
                if turn_begin is False:
                    end_desat.append(i)
                    minmax_desat.append(begin_desat[-1] + np.argmin(signal[begin_desat[-1]: end_desat[-1]]))
                    turn_begin = True
                    
    elif upperbound:
        for i in range(len(signal)):
            if i == 0:
                continue
            if np.isnan(signal[i]):  
                continue
            if (signal[i - 1] <= threshold_value) and (signal[i] > threshold_value):
                if turn_begin is True:
                    begin_desat.append(i)
                    turn_begin = False
            if (signal[i - 1] > threshold_value) and (signal[i] <= threshold_value):
                if turn_begin is False:
                    end_desat.append(i)
                    minmax_desat.append(begin_desat[-1] + np.argmax(signal[begin_desat[-1]: end_desat[-1]]))
                    turn_begin = True

    if turn_begin is False:
        begin_desat = begin_desat[0:-1]

    begin_desat = np.array(begin_desat).astype(int)
    end_desat   = np.array(end_desat).astype(int)

    ODI = len(begin_desat) / len(signal) * 12 
    
    return ODI, begin_desat, end_desat, minmax_desat

In [36]:
def DesaturationsMeasures(signal, threshold_method= 'Quantile', 
                          hard_threshold= 90, quantile_threshold= 0.10,
                          desat_min_length= 1, desat_max_length= 3, min_dist_meta_event= 1,
                          lowerbound= True, upperbound= False):

#     assert hard_threshold   > 0
    assert desat_max_length > 0
        
    check_shape(signal)
    
    warnings.simplefilter('ignore', np.RankWarning)
    warnings.filterwarnings("ignore", category=RuntimeWarning)
                                                                       
    if (threshold_method == 'Hard') or (threshold_method == 'Quantile'):
        ODI, begin_desat, end_desat, minmax_desat = hard_threshold_detector(signal, threshold_method, 
                                                                            hard_threshold, quantile_threshold,
                                                                            lowerbound, upperbound)

    counter_desat, begin_desat, end_desat, minmax_desat = group_meta_desat(signal, min_dist_meta_event, 
                                                                           begin_desat, end_desat, minmax_desat,
                                                                           lowerbound, upperbound)

    counter_desat, begin_desat, end_desat, minmax_desat = remove_nan_desats(signal, begin_desat, end_desat,
                                                                            minmax_desat, counter_desat)    

    counter_desat, begin_desat, end_desat, minmax_desat = remove_small_desats(counter_desat, begin_desat, 
                                                                              end_desat, minmax_desat, 
                                                                              desat_min_length)

    measures = get_desaturation_features(signal, begin_desat, end_desat, minmax_desat, lowerbound, upperbound)

    return measures

### Hypoxic Burden Measures

In [37]:
def comp_ca(signal, CA_Baseline, lowerbound= True, upperbound= False):

    if CA_Baseline is None:
        CA_Baseline = np.nanmean(signal)

    if lowerbound: 
        res = 0
        for value in signal:
            if value < CA_Baseline:
                res += CA_Baseline - value
                
    if upperbound: 
        res = 0
        for value in signal:
            if value > CA_Baseline:
                res += value - CA_Baseline

    return res / len(signal)

In [38]:
def comp_ct(signal, CT_Threshold, lowerbound= True, upperbound= False):

    if lowerbound:
        with np.errstate(invalid='ignore'):
            return 100 * len(signal[signal <= CT_Threshold]) / len(signal)
        
    elif upperbound:
        with np.errstate(invalid='ignore'):
            return 100 * len(signal[signal >= CT_Threshold]) / len(signal)

In [39]:
def HypoxicBurdenMeasures(signal, begin, end, CT_Threshold= 90, CA_Baseline= None,
                          lowerbound= True, upperbound= False):

    if isinstance(begin, int):
        begin = np.array([begin])
    if isinstance(end, int):
        end = np.array([end])

    assert len(begin) == len(end)

    check_shape(signal)
    
    desaturations, desaturation_valid, desaturation_length_all, desaturation_int_100_all, \
    desaturation_int_max_all, _, _, _, _, _ = desat_embedding(begin, end, end)

    time_spo2_array = np.array(range(len(signal)))
    
    for (i, desaturation) in enumerate(desaturations):
        desaturation_idx = (time_spo2_array >= desaturation['Start']) & (time_spo2_array <= desaturation['End'])

        if np.sum(desaturation_idx) == 0:
            continue

        signal = np.array(signal)

        desaturation_spo2 = signal[desaturation_idx]
        desaturation_max  = np.nanmax(desaturation_spo2)

        desaturation_valid[i] = True
        desaturation_length_all[i]  = desaturation['Duration']
        desaturation_int_100_all[i] = np.nansum(100 - desaturation_spo2)
        desaturation_int_max_all[i] = np.nansum(desaturation_max - desaturation_spo2)        

    if np.sum(desaturation_valid) != 0:
        POD    = np.nansum(desaturation_length_all[desaturation_valid])  / len(signal)
        AODmax = np.nansum(desaturation_int_max_all[desaturation_valid]) / len(signal)
        AOD100 = np.nansum(desaturation_int_100_all[desaturation_valid]) / len(signal)
    else:
        POD = 0
        AODmax = 0
        AOD100 = 0
        
    CA = comp_ca(signal, CA_Baseline , lowerbound, upperbound)
    CT = comp_ct(signal, CT_Threshold, lowerbound, upperbound)
    
    CA  = np.round(CA, 2)
    CT  = np.round(CT, 2)
    POD = np.round(POD, 2)
    AODmax = np.round(AODmax, 2)
    AOD100 = np.round(AOD100, 2)

    if lowerbound:
        measures = {
                    'LP_CA' : CA,
                    'LP_CT' : CT,
                    'LP_POD': POD,
                    'LP_AODmax' : AODmax,
                    'LP_AOD100' : AOD100
                   }
        
    if upperbound:
        measures = {
                    'UP_CA' : CA,
                    'UP_CT' : CT,
                    'UP_POD': POD,
                    'UP_AODmax' : AODmax,
                    'UP_AOD100' : AOD100
                   }
    
    return measures

### Fourier Transform

In [40]:
def FFTMeasures(signal, window_size):
    
    missing_count = np.sum(np.isnan(signal))
    non_missing_count = len(signal) - missing_count
    
    if non_missing_count > 2:
    
        sampling_rate = 0.2

        signal = pd.Series(signal)
        signals_filled = signal.interpolate(method='polynomial', order=1)
        signals_filled = signals_filled.fillna(method='ffill')
        signals_filled = signals_filled.fillna(method='bfill')
        signals_filled = np.array(signals_filled)
        
        signals_filled = signals_filled - np.mean(signals_filled)

        fft_result = fft(signals_filled)

        magnitude_spectrum = np.round(np.abs(fft_result), 3)

        peak_freq = np.argmax(magnitude_spectrum) * (sampling_rate / window_size)
        spectral_centroid = librosa.feature.spectral_centroid(y=signals_filled, sr=sampling_rate)[0][0]
        spectral_bandwidth= librosa.feature.spectral_bandwidth(y=signals_filled, sr=sampling_rate)[0][0]
        spectral_flatness = librosa.feature.spectral_flatness(y=signals_filled)[0][0]
        spectral_energy   = np.sum(magnitude_spectrum**2)
        spectral_entropy  = -np.sum((magnitude_spectrum / np.sum(magnitude_spectrum)) * np.log2(magnitude_spectrum / np.sum(magnitude_spectrum) + 1e-12))

        measures = {
                    'FFT_peak_freq': np.round(peak_freq, 3),
                    'FFT_spec_centroid' : np.round(spectral_centroid, 3),
                    'FFT_spec_bandwidth': np.round(spectral_bandwidth, 3),
                    'FFT_spec_flatness' : np.round(spectral_flatness, 3),
                    'FFT_spec_energy' : np.round(spectral_energy, 3),
                    'FFT_spec_entropy': np.round(spectral_entropy, 3)
                   }
        
    else:
        
        measures = {
                    'FFT_peak_freq': 0,
                    'FFT_spec_centroid' : 0,
                    'FFT_spec_bandwidth': 0,
                    'FFT_spec_flatness' : 0,
                    'FFT_spec_energy' : 0,
                    'FFT_spec_entropy': 0
                   }

    return measures

### Short-Time Fourier Transform (STFT)

In [41]:
def STFTMeasures(signal):
    
    missing_count = np.sum(np.isnan(signal))
    non_missing_count = len(signal) - missing_count
    
    if non_missing_count > 2:
    
        signal = pd.Series(signal)
        signals_filled = signal.interpolate(method='polynomial', order=1)
        signals_filled = signals_filled.fillna(method='ffill')
        signals_filled = signals_filled.fillna(method='bfill')
        signals_filled = np.array(signals_filled)
        
        signals_filled = signals_filled - np.mean(signals_filled)

        f, t, Zxx = stft(signals_filled, fs=0.2)  

        spectral_centroids = librosa.feature.spectral_centroid(S=np.abs(Zxx))[0]
        spectral_bandwidth = librosa.feature.spectral_bandwidth(S=np.abs(Zxx))[0]
        spectral_flatness  = librosa.feature.spectral_flatness(S=np.abs(Zxx))[0]
        spectral_rolloff   = librosa.feature.spectral_rolloff(S=np.abs(Zxx))[0]

        measures = {
                    'STFT_spec_centroid' : np.round(np.mean(spectral_centroids), 3),
                    'STFT_spec_bandwidth': np.round(np.mean(spectral_bandwidth), 3),
                    'STFT_spec_flatness' : np.round(np.mean(spectral_flatness), 3),
                    'STFT_spec_rolloff'  : np.round(np.mean(spectral_rolloff), 3),
                   }
    else:
        
        measures = {
                    'STFT_spec_centroid' : 0,
                    'STFT_spec_bandwidth': 0,
                    'STFT_spec_flatness' : 0,
                    'STFT_spec_rolloff'  : 0,
                   }
    
    return measures

### Discrete Wavelet Transform (DWT)

In [42]:
def missing_wavelet_feature():

    features = {}

    features['DWT_mean_approx'] = 0
    features['DWT_median_approx'] = 0
    features['DWT_variance_approx'] = 0
    features['DWT_std_approx'] = 0
    features['DWT_skewness_approx'] = 0
    features['DWT_kurtosis_approx'] = 0
    features['DWT_energy_approx']   = 0
    features['DWT_entropy_approx']  = 0
    
    for i in [1, 2]: 
        features[f'DWT_mean_detail_{i}'] = 0
        features[f'DWT_median_detail_{i}'] = 0
        features[f'DWT_variance_detail_{i}'] = 0
        features[f'DWT_std_detail_{i}'] = 0
        features[f'DWT_skewness_detail_{i}'] = 0
        features[f'DWT_kurtosis_detail_{i}'] = 0
        features[f'DWT_energy_detail_{i}']   = 0
        features[f'DWT_entropy_detail_{i}']  = 0
        
    return features

In [43]:
def safe_entropy(coeffs):
    
    coeffs_norm = coeffs / np.linalg.norm(coeffs)
    coeffs_norm = np.where(coeffs_norm == 0, 1e-12, coeffs_norm)  
    entropy = -np.sum(coeffs_norm**2 * np.log2(np.abs(coeffs_norm) + 1e-12))
    
    return entropy

In [44]:
def wavelet_feature_extraction(coeffs):

    features = {}
    
    approx_coeffs = coeffs[0]
    features['DWT_mean_approx'] = np.round(np.mean(approx_coeffs), 3)
    features['DWT_median_approx'] = np.round(np.median(approx_coeffs), 3)
    features['DWT_variance_approx'] = np.round(np.var(approx_coeffs), 3)
    features['DWT_std_approx'] = np.round(np.std(approx_coeffs), 3)
    features['DWT_skewness_approx'] = np.round(skew(approx_coeffs), 3)
    features['DWT_kurtosis_approx'] = np.round(kurtosis(approx_coeffs), 3)
    features['DWT_energy_approx']   = np.round(np.sum(approx_coeffs ** 2), 3)
    features['DWT_entropy_approx']  = np.round(safe_entropy(approx_coeffs), 3)
    
    for i, detail_coeffs in enumerate(coeffs[1:], 1): 
        features[f'DWT_mean_detail_{i}'] = np.round(np.mean(detail_coeffs), 3)
        features[f'DWT_median_detail_{i}'] = np.round(np.median(detail_coeffs), 3)
        features[f'DWT_variance_detail_{i}'] = np.round(np.var(detail_coeffs), 3)
        features[f'DWT_std_detail_{i}'] = np.round(np.std(detail_coeffs), 3)
        features[f'DWT_skewness_detail_{i}'] = np.round(skew(detail_coeffs), 3)
        features[f'DWT_kurtosis_detail_{i}'] = np.round(kurtosis(detail_coeffs), 3)
        features[f'DWT_energy_detail_{i}']   = np.round(np.sum(detail_coeffs ** 2), 3)
        features[f'DWT_entropy_detail_{i}']  = np.round(safe_entropy(detail_coeffs), 3)
        
    return features


def WaveletMeasures(signal):
    
    missing_count = np.sum(np.isnan(signal))
    non_missing_count = len(signal) - missing_count
    
    if non_missing_count > 2:
    
        signal = pd.Series(signal)
        signals_filled = signal.interpolate(method='polynomial', order=1)
        signals_filled = signals_filled.fillna(method='ffill')
        signals_filled = signals_filled.fillna(method='bfill')
        signals_filled = np.array(signals_filled)
        
        coeffs = pywt.wavedec(signals_filled, 'db4', mode='symmetric', level=2)

        measures = wavelet_feature_extraction(coeffs)
        
    else:
        
        measures = missing_wavelet_feature()
    
    return measures

In [45]:
def WaveletMeasures(signal):
    
    missing_count = np.sum(np.isnan(signal))
    non_missing_count = len(signal) - missing_count
    
    if non_missing_count > 2:
    
        signal = pd.Series(signal)
        signals_filled = signal.interpolate(method='polynomial', order=1)
        signals_filled = signals_filled.fillna(method='ffill')
        signals_filled = signals_filled.fillna(method='bfill')
        signals_filled = np.array(signals_filled)
        
        coeffs = pywt.wavedec(signals_filled, 'db4', mode='symmetric', level=2)

        measures = wavelet_feature_extraction(coeffs)
        
    else:
        
        measures = missing_wavelet_feature()
    
    return measures

### Apply Rolling Window

In [46]:
def rolling_window_apply(signal, datetime_values, window_length, overlap, variable, lower_thresh, upper_thresh):

    if len(signal) < window_length:
        step = 1
        end_point = 1
        if len(signal) < 12:
            time_slot = 60
        elif len(signal) < 24:
            time_slot = 120
        else:
            time_slot = 180
    else:
        step = window_length - overlap
        end_point = len(signal) - window_length + 1
    
    results = []
    
    for start in range(0, end_point, step):
        
        if len(signal) < window_length:
            window_data = signal[:]
            window_data_length = len(window_data)
            window_results = {}
            window_results = {'datetime': time_slot}
            
        else:
            window_data = signal[start:start + window_length]
            window_data_length = len(window_data)
            window_results = {}
            window_results = {'datetime': datetime_values[start + window_length - 1] + 5}
                
        OverallGeneral_measures = OverallGeneralMeasures(window_data, 
                                                         ZC_Baseline= None, 
                                                         M_Threshold= 2, 
                                                         DI_Window= int(window_data_length/3))

        PRSAMeasures_measures = PRSAMeasures(window_data, 
                                             PRSA_Window= int(window_data_length/3),
                                             K_AC= 2)
        
        PSDMeasures_measures = PSDMeasures(window_data, 
                                           frequency_low_threshold= 0.003, 
                                           frequency_high_threshold= 0.042)

        Complexity_measures = ComplexityMeasures(window_data, 
                                                 CTM_Threshold= 2, 
                                                 DFA_Window= int(window_data_length/3), 
                                                 M_Sampen= 2, 
                                                 R_Sampen= 0.2, 
                                                 M_ApEn= 2, 
                                                 R_ApEn= 0.2)
        
        if variable != 'TMP':
        
            Abnormality_measures_LP = DesaturationsMeasures(window_data, 
                                                            threshold_method= 'Quantile',
                                                            hard_threshold= lower_thresh, 
                                                            quantile_threshold= 0.15,
                                                            desat_min_length= 1, 
                                                            desat_max_length= 3, 
                                                            min_dist_meta_event= 1,
                                                            lowerbound= True,
                                                            upperbound= False)

            HypoxicBurden_measures_LP = HypoxicBurdenMeasures(window_data, 
                                                              Abnormality_measures_LP['LP_begin'], 
                                                              Abnormality_measures_LP['LP_end'],
                                                              CT_Threshold= lower_thresh, 
                                                              CA_Baseline= None,
                                                              lowerbound= True,
                                                              upperbound= False)
        
        if variable != 'SPO2':
            
            Abnormality_measures_UP = DesaturationsMeasures(window_data, 
                                                            threshold_method= 'Quantile',
                                                            hard_threshold= upper_thresh, 
                                                            quantile_threshold= 0.85,
                                                            desat_min_length= 1, 
                                                            desat_max_length= 3, 
                                                            min_dist_meta_event= 1,
                                                            lowerbound= False,
                                                            upperbound= True)

            HypoxicBurden_measures_UP = HypoxicBurdenMeasures(window_data, 
                                                              Abnormality_measures_UP['UP_begin'], 
                                                              Abnormality_measures_UP['UP_end'],
                                                              CT_Threshold= upper_thresh, 
                                                              CA_Baseline= None,
                                                              lowerbound= False,
                                                              upperbound= True)
            
        FFT_measures = FFTMeasures(window_data, window_length)
        
        STFT_measures = STFTMeasures(window_data) 

        Wavelet_measures = WaveletMeasures(window_data)
        
        window_results.update(OverallGeneral_measures)
        window_results.update(PRSAMeasures_measures)
        window_results.update(PSDMeasures_measures)
        window_results.update(Complexity_measures)
        
        if variable != 'TMP':
            window_results.update(Abnormality_measures_LP)
            window_results.update(HypoxicBurden_measures_LP)
        
        if variable != 'SPO2':
            window_results.update(Abnormality_measures_UP)
            window_results.update(HypoxicBurden_measures_UP)
            
        window_results.update(FFT_measures)
        window_results.update(STFT_measures)
        window_results.update(Wavelet_measures)
        
        results.append(window_results)
        
    results = pd.DataFrame(results)

    return results

### Vital Sign Processing

In [47]:
def vital_process(stay_dir):
    
    dn = os.path.join(path_timeseries, stay_dir)
    
    try:
        sys.stdout.flush()
        
        fields = ['patientid', 'datetime', 'SpO2', 'Heart Rate', 'Respiratory Rate', 'Temperature Central',
                  'Invasive systolic arterial pressure', 'Invasive diastolic arterial pressure',
                  'Invasive mean arterial pressure', 'ST1 (ECG ST elevation)', 'ST2 (ECG ST elevation)', 
                  'ST3 (ECG ST elevation)']

        raw_vitals = dataframe_from_csv(os.path.join(path_timeseries, stay_dir, 'imputed_raw_timeseries_05min.csv'), fields=fields)

        datetime_data = raw_vitals['datetime'].values

        spO2_data = raw_vitals['SpO2'].values
        hr_data   = raw_vitals['Heart Rate'].values
        rr_data   = raw_vitals['Respiratory Rate'].values
        tmp_data  = raw_vitals['Temperature Central'].values
        ibps_data = raw_vitals['Invasive systolic arterial pressure'].values
        ibpd_data = raw_vitals['Invasive diastolic arterial pressure'].values
        ibpm_data = raw_vitals['Invasive mean arterial pressure'].values
        st1_data  = raw_vitals['ST1 (ECG ST elevation)'].values
        st2_data  = raw_vitals['ST2 (ECG ST elevation)'].values
        st3_data  = raw_vitals['ST3 (ECG ST elevation)'].values

        spO2_results = rolling_window_apply(spO2_data, datetime_data, window_length=72, overlap=60, 
                                            variable='SPO2', lower_thresh=92  , upper_thresh=None)
        hr_results   = rolling_window_apply(hr_data,   datetime_data, window_length=72, overlap=60, 
                                            variable='HR'  , lower_thresh=70  , upper_thresh=110)
        rr_results   = rolling_window_apply(rr_data,   datetime_data, window_length=72, overlap=60, 
                                            variable='RR'  , lower_thresh=12  , upper_thresh=18)
        tmp_results  = rolling_window_apply(tmp_data,  datetime_data, window_length=72, overlap=60, 
                                            variable='TMP' , lower_thresh=None, upper_thresh=38)
        ibps_results = rolling_window_apply(ibps_data, datetime_data, window_length=72, overlap=60, 
                                            variable='IBPS', lower_thresh=95  , upper_thresh=125)
        ibpd_results = rolling_window_apply(ibpd_data, datetime_data, window_length=72, overlap=60, 
                                            variable='IBPD', lower_thresh=75  , upper_thresh=85)
        ibpm_results = rolling_window_apply(ibpm_data, datetime_data, window_length=72, overlap=60, 
                                            variable='IBPM', lower_thresh=85  , upper_thresh=105)
        st1_results  = rolling_window_apply(st1_data , datetime_data, window_length=72, overlap=60, 
                                            variable='ST1' , lower_thresh=-0.5 , upper_thresh=0.8)
        st2_results  = rolling_window_apply(st2_data , datetime_data, window_length=72, overlap=60, 
                                            variable='ST2' , lower_thresh=-0.5 , upper_thresh=0.8)
        st3_results  = rolling_window_apply(st3_data , datetime_data, window_length=72, overlap=60, 
                                            variable='ST3' , lower_thresh=-0.5 , upper_thresh=0.8)

        columns_spO2 = spO2_results.columns.tolist()
        columns_tmp  = tmp_results.columns.tolist()
        columns_rest = hr_results.columns.tolist()

        spO2_columns = [columns_spO2[0]] + ['spo2_' + col for col in columns_spO2[1:]]
        hr_columns   = [columns_rest[0]] + ['hr_'   + col for col in columns_rest[1:]]
        rr_columns   = [columns_rest[0]] + ['rr_'   + col for col in columns_rest[1:]]
        tmp_columns  = [columns_tmp[0]]  + ['tmp_'  + col for col in columns_tmp[1:]]
        ibps_columns = [columns_rest[0]] + ['ibps_' + col for col in columns_rest[1:]]
        ibpd_columns = [columns_rest[0]] + ['ibpd_' + col for col in columns_rest[1:]]
        ibpm_columns = [columns_rest[0]] + ['ibpm_' + col for col in columns_rest[1:]]
        st1_columns  = [columns_rest[0]] + ['st1_'  + col for col in columns_rest[1:]]
        st2_columns  = [columns_rest[0]] + ['st2_'  + col for col in columns_rest[1:]]
        st3_columns  = [columns_rest[0]] + ['st3_'  + col for col in columns_rest[1:]]

        spO2_results.columns = spO2_columns
        hr_results.columns   = hr_columns
        rr_results.columns   = rr_columns
        tmp_results.columns  = tmp_columns
        ibps_results.columns = ibps_columns
        ibpd_results.columns = ibpd_columns
        ibpm_results.columns = ibpm_columns
        st1_results.columns  = st1_columns
        st2_results.columns  = st2_columns
        st3_results.columns  = st3_columns

        merged_df = spO2_results.merge(hr_results, on='datetime', how='inner')
        merged_df = merged_df.merge(rr_results,    on='datetime', how='inner')
        merged_df = merged_df.merge(tmp_results ,  on='datetime', how='inner')
        merged_df = merged_df.merge(ibps_results,  on='datetime', how='inner')
        merged_df = merged_df.merge(ibpd_results,  on='datetime', how='inner')
        merged_df = merged_df.merge(ibpm_results,  on='datetime', how='inner')
        merged_df = merged_df.merge(st1_results,   on='datetime', how='inner')
        merged_df = merged_df.merge(st2_results,   on='datetime', how='inner')
        merged_df = merged_df.merge(st3_results,   on='datetime', how='inner')
        merged_df['patientid'] = int(stay_dir)
        
#         merged_df.to_csv(os.path.join(dn, 'vitals_processed.csv'), index=False)
        merged_df.to_csv(os.path.join(dn, 'vitals_processed_6hours.csv'), index=False)
            
    except Exception as e:
        print(f"Error processing {stay_dir}: {e}")
        exception_stayID.append(stay_dir)

In [48]:
dirs = os.listdir(path_timeseries)

exception_stayID = []
num_processes = cpu_count()

In [ ]:
with Pool(num_processes) as p:
    for _ in tqdm(p.imap(vital_process, dirs), total=len(dirs)):
        pass

### Test

In [47]:
stay_dir = str(100)

fields = ['patientid', 'datetime', 'SpO2', 'Heart Rate', 'Respiratory Rate', 'Temperature Central',
          'Invasive systolic arterial pressure', 'Invasive diastolic arterial pressure',
          'Invasive mean arterial pressure', 'ST1 (ECG ST elevation)', 'ST2 (ECG ST elevation)', 
          'ST3 (ECG ST elevation)']

raw_vitals = dataframe_from_csv(os.path.join(path_timeseries, stay_dir, 'imputed_raw_timeseries_05min.csv'), fields=fields)

datetime_data = raw_vitals['datetime'].values

spO2_data = raw_vitals['SpO2'].values
hr_data   = raw_vitals['Heart Rate'].values
rr_data   = raw_vitals['Respiratory Rate'].values
tmp_data  = raw_vitals['Temperature Central'].values
ibps_data = raw_vitals['Invasive systolic arterial pressure'].values
ibpd_data = raw_vitals['Invasive diastolic arterial pressure'].values
ibpm_data = raw_vitals['Invasive mean arterial pressure'].values
st1_data  = raw_vitals['ST1 (ECG ST elevation)'].values
st2_data  = raw_vitals['ST2 (ECG ST elevation)'].values
st3_data  = raw_vitals['ST3 (ECG ST elevation)'].values

spO2_results = rolling_window_apply(spO2_data, datetime_data, window_length=36, overlap=24, 
                                    variable='SPO2', lower_thresh=92  , upper_thresh=None)
hr_results   = rolling_window_apply(hr_data,   datetime_data, window_length=36, overlap=24, 
                                    variable='HR'  , lower_thresh=70  , upper_thresh=110)
rr_results   = rolling_window_apply(rr_data,   datetime_data, window_length=36, overlap=24, 
                                    variable='RR'  , lower_thresh=12  , upper_thresh=18)
tmp_results  = rolling_window_apply(tmp_data,  datetime_data, window_length=36, overlap=24, 
                                    variable='TMP' , lower_thresh=None, upper_thresh=38)
ibps_results = rolling_window_apply(ibps_data, datetime_data, window_length=36, overlap=24, 
                                    variable='IBPS', lower_thresh=95  , upper_thresh=125)
ibpd_results = rolling_window_apply(ibpd_data, datetime_data, window_length=36, overlap=24, 
                                    variable='IBPD', lower_thresh=75  , upper_thresh=85)
ibpm_results = rolling_window_apply(ibpm_data, datetime_data, window_length=36, overlap=24, 
                                    variable='IBPM', lower_thresh=85  , upper_thresh=105)
st1_results = rolling_window_apply(st1_data, datetime_data, window_length=36, overlap=24, 
                                    variable='ST1', lower_thresh=-0.5  , upper_thresh=0.8)
st2_results = rolling_window_apply(st2_data, datetime_data, window_length=36, overlap=24, 
                                    variable='ST2', lower_thresh=-0.5  , upper_thresh=0.8)
st3_results = rolling_window_apply(st3_data, datetime_data, window_length=36, overlap=24, 
                                    variable='ST3', lower_thresh=-0.5  , upper_thresh=0.8)

columns_spO2 = spO2_results.columns.tolist()
columns_tmp  = tmp_results.columns.tolist()
columns_rest = hr_results.columns.tolist()

spO2_columns = [columns_spO2[0]] + ['spo2_' + col for col in columns_spO2[1:]]
hr_columns   = [columns_rest[0]] + ['hr_'   + col for col in columns_rest[1:]]
rr_columns   = [columns_rest[0]] + ['rr_'   + col for col in columns_rest[1:]]
tmp_columns  = [columns_tmp[0]]  + ['tmp_'  + col for col in columns_tmp[1:]]
ibps_columns = [columns_rest[0]] + ['ibps_' + col for col in columns_rest[1:]]
ibpd_columns = [columns_rest[0]] + ['ibpd_' + col for col in columns_rest[1:]]
ibpm_columns = [columns_rest[0]] + ['ibpm_' + col for col in columns_rest[1:]]
st1_columns  = [columns_rest[0]] + ['st1_'  + col for col in columns_rest[1:]]
st2_columns  = [columns_rest[0]] + ['st2_'  + col for col in columns_rest[1:]]
st3_columns  = [columns_rest[0]] + ['st3_'  + col for col in columns_rest[1:]]

spO2_results.columns = spO2_columns
hr_results.columns   = hr_columns
rr_results.columns   = rr_columns
tmp_results.columns  = tmp_columns
ibps_results.columns = ibps_columns
ibpd_results.columns = ibpd_columns
ibpm_results.columns = ibpm_columns
st1_results.columns  = st1_columns
st2_results.columns  = st2_columns
st3_results.columns  = st3_columns

merged_df = spO2_results.merge(hr_results, on='datetime', how='inner')
merged_df = merged_df.merge(rr_results,    on='datetime', how='inner')
merged_df = merged_df.merge(tmp_results ,  on='datetime', how='inner')
merged_df = merged_df.merge(ibps_results,  on='datetime', how='inner')
merged_df = merged_df.merge(ibpd_results,  on='datetime', how='inner')
merged_df = merged_df.merge(ibpm_results,  on='datetime', how='inner')
merged_df = merged_df.merge(st1_results,   on='datetime', how='inner')
merged_df = merged_df.merge(st2_results,   on='datetime', how='inner')
merged_df = merged_df.merge(st3_results,   on='datetime', how='inner')
merged_df['patientid'] = stay_dir

In [48]:
PRSA_cols = ['PRSAc', 'PRSAad', 'PRSAos', 'PRSAsb', 'PRSAsa', 'AC']
PRSA_selected_cols = [col for col in merged_df.columns if any(keyword in col for keyword in PRSA_cols)]
PRSA_df = merged_df[PRSA_selected_cols]


PSD_cols = ['PSD_total', 'PSD_band', 'PSD_ratio', 'PSD_peak']
PSD_selected_cols = [col for col in merged_df.columns if any(keyword in col for keyword in PSD_cols)]
PSD_df = merged_df[PSD_selected_cols]


COMP_cols = ['ApEn', 'LZ', 'CTM', 'SampEn', 'DFA']
COMP_selected_cols = [col for col in merged_df.columns if any(keyword in col for keyword in COMP_cols)]
COMP_df = merged_df[COMP_selected_cols]


DES_cols = ['ODI', 'DL_u', 'DL_sd', 'DA100_u', 'DA100_sd', 'DAmax_u', 'DAmax_sd', 'DD100_u', 'DD100_sd', 'DDmax_u',
            'DDmax_sd', 'DS_u', 'DS_sd', 'TD_u', 'TD_sd', 'DL_a_u', 'DL_a_sd', 'DL_b_u', 'DL_b_sd', 'begin', 'end' ]
DES_selected_cols = [col for col in merged_df.columns if any(keyword in col for keyword in DES_cols)]
DES_df = merged_df[DES_selected_cols]


HYP_cols = ['CA', 'CT', 'POD', 'AODmax', 'AOD100']
HYP_selected_cols = [col for col in merged_df.columns if any(keyword in col for keyword in HYP_cols)]
HYP_df = merged_df[HYP_selected_cols]


FF_cols = ['FFT', 'STFT', 'DWT']
FF_selected_cols = [col for col in merged_df.columns if any(keyword in col for keyword in FF_cols)]
FF_df = merged_df[FF_selected_cols]

In [ ]:
def plot_psd(signal, fs=0.2):
    
    signal = np.array(signal)
    signal = signal[~np.isnan(signal)]
    
    freq, power = welch(signal, fs=fs, window='hamming', nperseg=len(signal), noverlap=0)
    plt.semilogy(freq, power)
    plt.title('Power Spectral Density of SpO2 Signal')
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('PSD (dB/Hz)')
    plt.grid(True)

In [ ]:
first_window_signal  = rr_data[:24]  
middle_window_signal = rr_data[len(spO2_data)//2:len(spO2_data)//2+24]  
last_window_signal   = rr_data[-24:]  

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plot_psd(first_window_signal)

plt.subplot(1, 3, 2)
plot_psd(middle_window_signal)

plt.subplot(1, 3, 3)
plot_psd(last_window_signal)

plt.tight_layout()
plt.show()